NYC 311 Data Ingestion (Optimization Summary)

Data partitioned into 10-day chunks (90-day window) to reduce API load and improve reliability
Each chunk retrieved in 20K record batches using unique_key-based pagination to ensure ordering and avoid duplicates
Query design enforces deterministic sorting and controlled request sizes, mitigating API rate limits and timeouts

Limitation

Intermediate results stored in memory before aggregation, leading to higher RAM usage

Future Improvements

Stream-based processing (fetch → transform → store)
Parquet storage on S3 for scalability
Checkpointing for fault tolerance
Controlled parallel ingestion for improved throughput

In [1]:
from sodapy import Socrata
import pandas as pd
from datetime import datetime, timedelta
import time

client = Socrata("data.cityofnewyork.us", None,timeout = 60)

#cutoff = (datetime.now() - timedelta(days=10)).strftime("%Y-%m-%dT%H:%M:%S")
#current date - 90 days is cutoff

batch_size=20000
all_dfs = []

start_date = (datetime.now() - timedelta(days=90))

start_time = time.time()

for i in range(9):

    chunk_start = start_date + timedelta(days=i*10)
    chunk_end   = chunk_start + timedelta(days=10)

    print(f"Loading Batch :{i} of Data -> From {(str(chunk_start))} to {str(chunk_end)}")
    last_key = 0
    while True:               
        results = client.get(
            "erm2-nwe9",
            where = (f"""created_date >= '{chunk_start.strftime("%Y-%m-%dT%H:%M:%S")}' 
            AND created_date < '{chunk_end.strftime("%Y-%m-%dT%H:%M:%S")}'
            AND unique_key > '{last_key}' """ ),
            order = "unique_key ASC",
            limit = batch_size
        )
    
        if not results:
            break
        
        df_batch = pd.DataFrame(results)
        all_dfs.append(df_batch)

        last_key = df_batch["unique_key"].astype(int).max()
        print("\t",len(df_batch),"Loaded , Last Key : ", last_key)

start_time2= time.time()
dfs = pd.concat(all_dfs, ignore_index=True)
end_time2= time.time()
end_time = time.time()
print("Overall Time",end_time - start_time)
print("Concat Time",end_time2 - start_time2)
print(df_batch.shape)
print(len(all_dfs))
print(dfs.shape)

Loading Batch :0 of Data -> From 2026-03-24 11:51:02.097638 to 2026-04-03 11:51:02.097638
	 20000 Loaded , Last Key :  68459496
	 20000 Loaded , Last Key :  68480483
	 20000 Loaded , Last Key :  68500996
	 20000 Loaded , Last Key :  68522219
	 20000 Loaded , Last Key :  68543504
	 7365 Loaded , Last Key :  68614445
Loading Batch :1 of Data -> From 2026-04-03 11:51:02.097638 to 2026-04-13 11:51:02.097638
	 20000 Loaded , Last Key :  68571911
	 20000 Loaded , Last Key :  68593164
	 20000 Loaded , Last Key :  68614491
	 20000 Loaded , Last Key :  68635490
	 15827 Loaded , Last Key :  68745181
Loading Batch :2 of Data -> From 2026-04-13 11:51:02.097638 to 2026-04-23 11:51:02.097638
	 20000 Loaded , Last Key :  68673358
	 20000 Loaded , Last Key :  68694869
	 20000 Loaded , Last Key :  68715845
	 20000 Loaded , Last Key :  68736910
	 20000 Loaded , Last Key :  68758297
	 6013 Loaded , Last Key :  68843920
Loading Batch :3 of Data -> From 2026-04-23 11:51:02.097638 to 2026-05-03 11:51:02.097

In [2]:
import pyarrow 
import boto3
import s3fs



In [ ]:
for r in all_dfs :
    print(r.head(1))
    print(r.shape)
    break

In [8]:
pip install --upgrade pyarrow pandas


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

table = pa.Table.from_pandas(dfs)

pq.write_table(table, "raw_data.parquet")